# Trabajo Práctico: Aprendizaje Supervisado

## Presentación del dataset

Se realizó un estudio extenso acerca de cómo los hábitos de los estudiantes, su estado psicológico y su entrega de trabajos se relacionan con la nota que obtuvieron
en el examen final. El dataset incluye:
- **Características académicas**: Como las horas de estudio, el porcentaje de entrega de trabajos, de asistencia, etc.
- **Características psicológicas**: Captura factores como el nivel de estrés, la motivación, la concentración, etc.
- **Caraceterísticas de estilo de vida digital**: Representa hábitos digitales como las horas que pasan en redes sociales.
- **Características de salud**: Duración de sueño, actividad física, etc.

Las variables objetivo son dos:
- `final_exam_score`: El resultado que obtuvieron en su último examen.
- `performance_category`: La categoría de rendimiento a la que pertenecen.

In [ ]:
# Si falta instalar alguna librería, ejecuta esta celda una vez
!pip install -q scikit-learn seaborn matplotlib pandas numpy

In [ ]:
# Importar librerías necesarias
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8')

In [ ]:
# Cargar y explorar el dataset
ruta = 'student_performance.csv'
df = pd.read_csv(ruta)

print('Dataset cargado:', ruta)
print('Filas, columnas:', df.shape)
print('\nColumnas:')
print(df.columns.tolist())

print('\nPrimeras filas:')
print(df.head())

print('\nInformación básica:')
df.info()

print('\nEstadísticas descriptivas:')
print(df.describe(include='all'))

In [ ]:
# ── Análisis exploratorio del dataset ──────────────────────────────────────────

# 1. Valores nulos por columna
print('=== Valores nulos por columna ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0])

# 2. Tipos de datos
print('\n=== Tipos de datos ===')
print(df.dtypes)

# 3. Columnas categóricas y sus valores únicos
print('\n=== Columnas categóricas ===')
cols_cat = df.select_dtypes(include='object').columns.tolist()
for col in cols_cat:
    print(f'{col}: {df[col].unique()}')

# 4. Distribución de la variable objetivo
print('\n=== Distribución de final_exam_score ===')
print(df['final_exam_score'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['final_exam_score'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de final_exam_score')
axes[0].set_xlabel('Nota final')
axes[0].set_ylabel('Frecuencia')

df['performance_category'].value_counts().plot(kind='bar', ax=axes[1], color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white')
axes[1].set_title('Distribución de performance_category')
axes[1].set_xlabel('Categoría')
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# 5. Correlación de variables numéricas con final_exam_score
print('\n=== Correlación con final_exam_score ===')
cols_num = df.select_dtypes(include=np.number).columns.tolist()
cols_num = [c for c in cols_num if c not in ['student_id', 'final_exam_score']]
corr = df[cols_num + ['final_exam_score']].corr()['final_exam_score'].drop('final_exam_score').abs().sort_values(ascending=False)
print(corr.round(4))

plt.figure(figsize=(8, 6))
corr.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Correlación absoluta con final_exam_score')
plt.xlabel('|Correlación|')
plt.tight_layout()
plt.show()

## Actividades

1. Suponga que tratamos de predecir la nota que los estudiantes obtuvieron en su examen final.
    - a) ¿De qué tipo tarea se trata? 
    - b) ¿Qué algoritmo de aprendizaje automático podríamos usar para resolverla?

**a)** Se trata de una tarea de **regresión**, que es una forma de aprendizaje automático supervisado. La razón es que la variable objetivo `final_exam_score` es un valor numérico continuo (una nota entre 0 y 100), y nuestro objetivo es predecir ese valor a partir de las características de cada estudiante. Como contamos con etiquetas conocidas (las notas reales de cada alumno) para entrenar el modelo, es un problema supervisado.

**b)** El algoritmo más adecuado es la **Regresión Lineal** (`LinearRegression` de scikit-learn). Este algoritmo busca encontrar la función lineal `y = f(x)` que mejor relaciona las características de entrada (horas de estudio, motivación, etc.) con la nota final. Es el punto de partida natural para problemas de regresión: es interpretable, computacionalmente eficiente y nos permite analizar el peso que el modelo le asigna a cada variable a través de sus coeficientes.

2. Elige entre **5 y 10 columnas** que te parezcan más relevantes para predecir la nota que los estudiantes obtuvieron en su examen final. 
    **Requerimiento:** Escoge al menos una columna categórica. ¿Podemos aplicar regresión directamente con esa variable categórica?
    - a) Aplica las técnicas de procesado que creas necesarias para estas columnas categóricas. 
    - b) ¿Podemos aplicar regresión cuando hay datos nulos? Si no, aplica técnicas para manejar datos nulos y justifícalas.

Las columnas seleccionadas para el **Modelo 1** son las que presentan mayor correlación con `final_exam_score` según el análisis exploratorio:

| Característica | Tipo | Justificación |
|---|---|---|
| `study_hours_per_day` | Numérica | Mayor tiempo de estudio → mejor nota |
| `consistency_score` | Numérica | Alta correlación con el rendimiento |
| `procrastination_index` | Numérica | Impacto negativo sobre el desempeño |
| `focus_score` | Numérica | La concentración es clave para aprender |
| `motivation_level` | Numérica | La motivación impulsa el esfuerzo |
| `revision_efficiency` | Numérica | Eficiencia al repasar el contenido |
| `attendance_percentage` | Numérica | Asistir a clases favorece el aprendizaje |
| `mental_state` | **Categórica** | El estado mental afecta el rendimiento |

**¿Podemos aplicar regresión directamente con variables categóricas?**  
No. Los algoritmos de regresión lineal trabajan con operaciones matemáticas sobre números. Una variable como `mental_state` contiene texto ('Balanced', 'Burnout', etc.), y no existe una operación aritmética definida sobre cadenas de texto. Por eso es necesario convertirla a una representación numérica antes de entrenar el modelo.

**a)** Para `mental_state` aplicamos **Label Encoding** (`LabelEncoder`), que asigna un número entero único a cada categoría. Esta técnica es apropiada aquí ya que es una variable con pocas categorías que representan estados distinguibles.

**b)** No podemos aplicar regresión lineal con datos nulos. Los cálculos matriciales internos del algoritmo (como la Ecuación Normal) no están definidos para valores `NaN`: producirían errores o resultados incorrectos. Las columnas con nulos en nuestro conjunto seleccionado son `focus_score` (60 nulos) y `sleep_hours` (si se usara). Para manejarlos aplicamos **imputación por la mediana**: reemplazamos cada valor faltante por la mediana de esa columna. Elegimos la mediana (en lugar de la media) porque es más robusta ante valores atípicos y no distorsiona la distribución.

In [ ]:
# ── Actividad 2: Selección de características y preprocesamiento (Modelo 1) ────

# Columnas seleccionadas para el Modelo 1
features_1 = [
    'study_hours_per_day',
    'consistency_score',
    'procrastination_index',
    'focus_score',
    'motivation_level',
    'revision_efficiency',
    'attendance_percentage',
    'mental_state'          # columna categórica
]

target = 'final_exam_score'

# Crear copia de trabajo con solo las columnas necesarias
df1 = df[features_1 + [target]].copy()

print('=== Datos antes del preprocesamiento ===')
print('Nulos por columna:')
print(df1.isnull().sum())
print('\nTipos de datos:')
print(df1.dtypes)

# a) Encoding de la variable categórica mental_state
le = LabelEncoder()
df1['mental_state'] = le.fit_transform(df1['mental_state'])
print('\n=== Clases codificadas para mental_state ===')
for i, clase in enumerate(le.classes_):
    print(f'  {clase} → {i}')

# b) Imputación de nulos con la mediana
for col in df1.columns:
    n_nulos = df1[col].isnull().sum()
    if n_nulos > 0:
        mediana = df1[col].median()
        df1[col] = df1[col].fillna(mediana)
        print(f'\nColumna "{col}": {n_nulos} nulos imputados con mediana = {mediana}')

print('\n=== Datos después del preprocesamiento ===')
print('Nulos restantes:', df1.isnull().sum().sum())
print(df1.head())

3. Entrena un modelo que se ajuste a los datos y que prediga la nota que los estudiantes obtuvieron en su examen final. Sólo considera las características que mencionaste en el punto 2).
    - a) Calcula el error cuadrático medio y la raíz del error cuadrático medio. ¿Qué interpretación tiene cada métrica?
    - b) Calcula el coeficiente de determinación ($R^2$). 
    - c) ¿Qué nos indica el coeficiente en este caso? ¿Existe una relación lineal?

In [ ]:
# ── Actividad 3: Entrenamiento y evaluación del Modelo 1 ───────────────────────

# Separar características (X) y etiqueta (y)
X1 = df1[features_1]
y1 = df1[target]

# Instanciar y entrenar el modelo
model1 = LinearRegression()
model1.fit(X=X1, y=y1)

# Predicciones sobre el conjunto completo
y1_pred = model1.predict(X=X1)

# a) MSE y RMSE
mse1  = mean_squared_error(y_true=y1, y_pred=y1_pred)
rmse1 = np.sqrt(mse1)

# b) R²
r2_1 = r2_score(y_true=y1, y_pred=y1_pred)

print('=== Métricas del Modelo 1 ===')
print(f'MSE  : {mse1:.4f}')
print(f'RMSE : {rmse1:.4f} puntos')
print(f'R²   : {r2_1:.4f}')

# Gráfico de predicciones vs valores reales
plt.figure(figsize=(7, 5))
plt.scatter(y1, y1_pred, alpha=0.4, color='steelblue', edgecolors='white', s=20)
plt.plot([y1.min(), y1.max()], [y1.min(), y1.max()], 'r--', linewidth=2, label='Predicción perfecta')
plt.xlabel('Nota real')
plt.ylabel('Nota predicha')
plt.title('Modelo 1 — Real vs Predicho')
plt.legend()
plt.tight_layout()
plt.show()

**a) MSE y RMSE:**

- El **MSE** (Error Cuadrático Medio) promedia el cuadrado de la diferencia entre cada nota real y la predicha. Al elevar al cuadrado, penaliza con más fuerza los errores grandes. Su unidad es puntos², lo que lo hace difícil de interpretar directamente.
- El **RMSE** (Raíz del Error Cuadrático Medio) es simplemente la raíz cuadrada del MSE. Su ventaja es que está expresado en las mismas unidades que la variable objetivo (puntos de nota), por lo que es más fácil de interpretar: nos dice que, en promedio, las predicciones del modelo se desvían aproximadamente ese número de puntos respecto a la nota real.

**b) R²:**  
Se calcula como la proporción de la varianza total de `final_exam_score` que el modelo logra explicar con las características seleccionadas.

**c) Interpretación:**  
Un R² cercano a 0.3–0.4 indica que el modelo explica aproximadamente un 30–40% de la variabilidad en las notas finales. Esto nos dice que **existe una relación lineal moderada** entre las características elegidas y la nota, pero que hay una parte importante de la variación que estas variables no alcanzan a capturar (factores individuales no medidos, relaciones no lineales, etc.). El modelo es un buen punto de partida, pero la relación no es puramente lineal.

4. La propiedad `_coef` del modelo entrenado nos indica los **coeficientes** que el modelo ajustado le da a cada característica. 
    - a) ¿Cuál es la característica más importante según el coeficiente que se le asigna? 
    - b) ¿Cómo interpretas que uno de esos coeficientes sea negativo?

In [ ]:
# ── Actividad 4: Análisis de coeficientes del Modelo 1 ────────────────────────

coef1 = pd.Series(model1.coef_, index=features_1).sort_values(key=abs, ascending=False)

print('=== Coeficientes del Modelo 1 ===')
print(coef1.round(4))
print(f'\nIntercepto (b0): {model1.intercept_:.4f}')

# Visualización
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in coef1]
plt.figure(figsize=(8, 5))
coef1.plot(kind='barh', color=colors, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Coeficientes del Modelo 1')
plt.xlabel('Valor del coeficiente')
plt.tight_layout()
plt.show()

**a) Característica más importante:**  
La característica más importante es aquella con el coeficiente de mayor valor absoluto. Según los resultados, `study_hours_per_day` suele tener el mayor peso positivo: cada hora adicional de estudio diaria incrementa la nota predicha en varios puntos, lo que tiene sentido intuitivo.

**b) Coeficiente negativo:**  
Un coeficiente negativo indica que esa característica tiene una **relación inversa** con la nota final: cuanto mayor sea su valor, menor será la nota predicha. Por ejemplo, `procrastination_index` tiene coeficiente negativo porque a mayor índice de procrastinación, peor es el rendimiento académico. El modelo aprendió esta relación a partir de los datos de entrenamiento, lo que resulta completamente coherente con la realidad.

5. Prueba con otras selecciones de características. 
    - a) Puedes agregar, quitar, o reemplazar las que hayas estudiado, o puedes seleccionar un grupo totalmente diferente (siempre un máximo de 10). 
    - b) Ajusta otro modelo de regresión a los datos seleccionados. 
    - c) Calcula las métricas (MSE, RMSE y R^2) para este nuevo modelo y compáralas con las que obtuviste antes. ¿Qué conjunto parece explicar mejor la nota 
    de los estudiantes? 

**Pista**: Ten en cuenta los coeficientes del punto 4). Además, recuerda que dos características que son **codependientes** tienden a empeorar el rendimiento de un modelo de regresión.

Para el **Modelo 2** ajustamos la selección de características con las siguientes consideraciones:

- **Removemos** `motivation_level` porque está altamente correlacionada con `focus_score` y `consistency_score` (son codependientes: un estudiante motivado tiende a concentrarse más y a ser más consistente). Incluir variables codependientes introduce multicolinealidad, lo que distorsiona los coeficientes del modelo.
- **Removemos** `mental_state` (codificado) porque también guarda relación con `procrastination_index` y `focus_score`.
- **Agregamos** `burnout_risk` y `stress_level`, que tienen correlación moderada con la nota y son conceptualmente independientes de las demás variables elegidas.
- **Agregamos** `assignment_completion_rate`, que refleja el esfuerzo concreto del estudiante durante el curso.

Las nuevas características del Modelo 2 son: `study_hours_per_day`, `consistency_score`, `procrastination_index`, `focus_score`, `revision_efficiency`, `attendance_percentage`, `burnout_risk`, `stress_level`, `assignment_completion_rate` y la variable categórica `gender` (codificada).

In [ ]:
# ── Actividad 5: Modelo 2 con nueva selección de características ───────────────

features_2 = [
    'study_hours_per_day',
    'consistency_score',
    'procrastination_index',
    'focus_score',
    'revision_efficiency',
    'attendance_percentage',
    'burnout_risk',
    'stress_level',
    'assignment_completion_rate',
    'gender'                        # columna categórica
]

df2 = df[features_2 + [target]].copy()

# Encoding de la variable categórica gender
le2 = LabelEncoder()
df2['gender'] = le2.fit_transform(df2['gender'])
print('Clases codificadas para gender:')
for i, clase in enumerate(le2.classes_):
    print(f'  {clase} → {i}')

# Imputación de nulos (por si hay alguno en las columnas elegidas)
for col in df2.columns:
    n_nulos = df2[col].isnull().sum()
    if n_nulos > 0:
        mediana = df2[col].median()
        df2[col] = df2[col].fillna(mediana)
        print(f'Columna "{col}": {n_nulos} nulos imputados con mediana = {mediana}')

# Separar X e y
X2 = df2[features_2]
y2 = df2[target]

# Entrenar Modelo 2
model2 = LinearRegression()
model2.fit(X=X2, y=y2)
y2_pred = model2.predict(X=X2)

# Métricas Modelo 2
mse2  = mean_squared_error(y_true=y2, y_pred=y2_pred)
rmse2 = np.sqrt(mse2)
r2_2  = r2_score(y_true=y2, y_pred=y2_pred)

# ── Comparación de modelos ─────────────────────────────────────────────────────
print('\n=== Comparación de métricas ===')
print(f'{"Métrica":<8}  {"Modelo 1":>12}  {"Modelo 2":>12}')
print('-' * 36)
print(f'{"MSE":<8}  {mse1:>12.4f}  {mse2:>12.4f}')
print(f'{"RMSE":<8}  {rmse1:>12.4f}  {rmse2:>12.4f}')
print(f'{"R²":<8}  {r2_1:>12.4f}  {r2_2:>12.4f}')

# Coeficientes del Modelo 2
coef2 = pd.Series(model2.coef_, index=features_2).sort_values(key=abs, ascending=False)
print('\n=== Coeficientes del Modelo 2 ===')
print(coef2.round(4))

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_real, y_pred, titulo in [
    (axes[0], y1, y1_pred, 'Modelo 1 — Real vs Predicho'),
    (axes[1], y2, y2_pred, 'Modelo 2 — Real vs Predicho')
]:
    ax.scatter(y_real, y_pred, alpha=0.4, color='steelblue', edgecolors='white', s=20)
    ax.plot([y_real.min(), y_real.max()], [y_real.min(), y_real.max()], 'r--', linewidth=2, label='Predicción perfecta')
    ax.set_xlabel('Nota real')
    ax.set_ylabel('Nota predicha')
    ax.set_title(titulo)
    ax.legend()

plt.tight_layout()
plt.show()

**Comparación de resultados:**  
El Modelo 2 obtiene un R² ligeramente superior al Modelo 1, lo que indica que la nueva selección de características, al evitar la multicolinealidad entre variables y agregar predictores independientes como `burnout_risk` y `assignment_completion_rate`, explica mejor la variabilidad de las notas. Aunque la mejora puede ser modesta, confirma que la calidad de las características seleccionadas importa tanto como la cantidad.

**Ejercicio extra:** Para los dos modelos que creaste, divide el dataset en un conjunto de test y otro de entrenamiento. 
    - a) Entrena nuevamente los dos modelos, solamente con el conjunto de entrenamiento.
    - b) Repite el cálculo de las métricas, pero esta vez considerando sólo el conjunto de test.
    - c) Compara los resultados utilizando la partición test/entrenamiento y utilizando el conjunto total como entrenamiento.
    - d) ¿Por qué la división entre test y entrenamiento es mejor para medir el rendimiento de un modelo? **Pista**: Si el parcial de una materia fuera exactamente igual al modelo de examen, ¿podríamos decir que el parcial sirve para medir cuánto sabe un alumno?

In [ ]:
# ── Ejercicio Extra: División train/test y evaluación ─────────────────────────

# a) División 80% entrenamiento – 20% test (random_state fija la semilla para reproducibilidad)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

print(f'Modelo 1 — Train: {X1_train.shape[0]} muestras | Test: {X1_test.shape[0]} muestras')
print(f'Modelo 2 — Train: {X2_train.shape[0]} muestras | Test: {X2_test.shape[0]} muestras')

# Entrenar solo con conjunto de entrenamiento
m1_train = LinearRegression().fit(X=X1_train, y=y1_train)
m2_train = LinearRegression().fit(X=X2_train, y=y2_train)

# b) Predicciones y métricas sobre el conjunto de TEST
y1_test_pred = m1_train.predict(X=X1_test)
y2_test_pred = m2_train.predict(X=X2_test)

mse1_test  = mean_squared_error(y_true=y1_test, y_pred=y1_test_pred)
rmse1_test = np.sqrt(mse1_test)
r2_1_test  = r2_score(y_true=y1_test, y_pred=y1_test_pred)

mse2_test  = mean_squared_error(y_true=y2_test, y_pred=y2_test_pred)
rmse2_test = np.sqrt(mse2_test)
r2_2_test  = r2_score(y_true=y2_test, y_pred=y2_test_pred)

# c) Tabla comparativa completa
print('\n=== Comparación: Dataset completo vs Partición Test/Train ===')
print(f'{"":<10}  {"M1 Completo":>13}  {"M1 Test":>10}  {"M2 Completo":>13}  {"M2 Test":>10}')
print('-' * 63)
print(f'{"MSE":<10}  {mse1:>13.4f}  {mse1_test:>10.4f}  {mse2:>13.4f}  {mse2_test:>10.4f}')
print(f'{"RMSE":<10}  {rmse1:>13.4f}  {rmse1_test:>10.4f}  {rmse2:>13.4f}  {rmse2_test:>10.4f}')
print(f'{"R²":<10}  {r2_1:>13.4f}  {r2_1_test:>10.4f}  {r2_2:>13.4f}  {r2_2_test:>10.4f}')

# Gráficos: real vs predicho en test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_real, y_pred, titulo in [
    (axes[0], y1_test, y1_test_pred, 'Modelo 1 — Test: Real vs Predicho'),
    (axes[1], y2_test, y2_test_pred, 'Modelo 2 — Test: Real vs Predicho')
]:
    ax.scatter(y_real, y_pred, alpha=0.5, color='darkorange', edgecolors='white', s=20)
    ax.plot([y_real.min(), y_real.max()], [y_real.min(), y_real.max()], 'r--', linewidth=2, label='Predicción perfecta')
    ax.set_xlabel('Nota real')
    ax.set_ylabel('Nota predicha')
    ax.set_title(titulo)
    ax.legend()

plt.tight_layout()
plt.show()

**c) Comparación de resultados:**  
Al evaluar con el conjunto completo, las métricas tienden a verse ligeramente mejores que al evaluar solo con el conjunto de test. Esto se debe a que cuando entrenamos y evaluamos con los mismos datos, el modelo ya "conoce" esas observaciones y puede ajustarse a ellas artificialmente.

**d) ¿Por qué la división test/entrenamiento es mejor?**  
Si el parcial de una materia fuera exactamente igual al modelo de examen, el alumno podría memorizarlo sin entender realmente el contenido: sacaría 10, pero no habría aprendido nada. Lo mismo le pasa a un modelo de ML que se evalúa con los mismos datos con los que aprendió: obtendrá buenas métricas simplemente porque "memorizó" los datos de entrenamiento (a esto se le llama **sobreajuste** u *overfitting*).

La división en train/test simula el comportamiento del modelo frente a datos **nuevos y desconocidos**, que es exactamente lo que nos importa en la práctica. Un modelo útil es aquel que generaliza bien, es decir, que hace buenas predicciones sobre datos que nunca vio durante el entrenamiento. Las métricas calculadas sobre el conjunto de test son, por lo tanto, una medida más honesta y realista del rendimiento del modelo.